In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/polprime/interaction-score0/eshop_clothing_clean.csv
/kaggle/input/datasets/polprime/clickstera/eshop_clothing_clean.csv


In [2]:
import pandas as pd
import numpy as np

df=pd.read_csv('/kaggle/input/datasets/polprime/clickstera/eshop_clothing_clean.csv', sep=';')
print(f"✅ Đọc dữ liệu xong: {df.shape[0]:,} dòng, {df.shape[1]} cột")

# TÍNH page_norm
# page gốc: 1→5 (cao = xem sâu = hứng thú)
# Chuẩn hóa Min-Max về [0.0, 1.0]
# page=1 → 0.0 | page=3 → 0.5 | page=5 → 1.0
df['page_norm'] = (df['page'] - 1) / (5 - 1)

print('page_norm - kiểm tra:')
print(df[['page', 'page_norm']].drop_duplicates().sort_values('page'))

# TÍNH order_norm
# order gốc: thứ tự click trong session (thấp = xem sớm = chú ý hơn)
# Phải đảo ngược + chuẩn hóa về [0.0, 1.0]
# session_size = 1 → gán 0.5 (Phương án B: trung tính)
df['order_rank']=df.groupby('session ID')['order'].rank(ascending=True)
df['session_size'] = df.groupby('session ID')['session ID'].transform('count')

df['order_norm']=np.where(
    df['session_size']>1,
    1-(df['order_rank']-1)/(df['session_size']-1),
    0.5
)

print('\norder_norm - kiểm tra session 1:')
print(df[df['session ID']==1][
      ['order', 'session_size', 'order_rank', 'order_norm']
].sort_values('order'))

# TÍNH interaction_score (50/50)
# Công thức: 0.5 × page_norm + 0.5 × order_norm
# Kết quả: [0.0, 1.0]
df['interaction_score']=0.5*df['page_norm']+0.5*df['order_norm']

print('interaction_score - thống kê:')
print(df['interaction_score'].describe().round(4))

'''# TẠO BINARY LABEL
# Ngưỡng = quantile 75%
# Top 25% score cao nhất → label = 1 (quan tâm cao)
# 75% còn lại           → label = 0 (quan tâm thấp)
threshold=df['interaction_score'].quantile(0.75)
df['label']=(df['interaction_score']>= threshold).astype(int)

print(f"\nNgưỡng phân loại: {threshold:.4f}")
print("\nPhân phối label:")
print(df['label'].value_counts())
print(df['label'].value_counts(normalize=True).round(3))'''

✅ Đọc dữ liệu xong: 165,474 dòng, 14 cột
page_norm - kiểm tra:
    page  page_norm
0      1       0.00
9      2       0.25
26     3       0.50
5      4       0.75
8      5       1.00

order_norm - kiểm tra session 1:
   order  session_size  order_rank  order_norm
0      1             9         1.0       1.000
1      2             9         2.0       0.875
2      3             9         3.0       0.750
3      4             9         4.0       0.625
4      5             9         5.0       0.500
5      6             9         6.0       0.375
6      7             9         7.0       0.250
7      8             9         8.0       0.125
8      9             9         9.0       0.000
interaction_score - thống kê:
count    165474.0000
mean          0.3388
std           0.1800
min           0.0000
25%           0.2143
50%           0.3500
75%           0.4821
max           1.0000
Name: interaction_score, dtype: float64


'# TẠO BINARY LABEL\n# Ngưỡng = quantile 75%\n# Top 25% score cao nhất → label = 1 (quan tâm cao)\n# 75% còn lại           → label = 0 (quan tâm thấp)\nthreshold=df[\'interaction_score\'].quantile(0.75)\ndf[\'label\']=(df[\'interaction_score\']>= threshold).astype(int)\n\nprint(f"\nNgưỡng phân loại: {threshold:.4f}")\nprint("\nPhân phối label:")\nprint(df[\'label\'].value_counts())\nprint(df[\'label\'].value_counts(normalize=True).round(3))'